# 2.1 KV Cache 优化

## 什么是KV Cache？

在自回归生成（如GPT系列模型）中，每生成一个新token都需要访问之前所有token的Key和Value向量。KV Cache将已计算的Key/Value向量缓存下来，避免重复计算，是LLM推理的核心优化。

## 为什么KV Cache是瓶颈？

KV Cache的内存占用与序列长度成正比：
$$M_{kv} = 2 \times n_{layers} \times n_{heads} \times d_{head} \times L \times b_{bytes}$$

其中：
- $n_{layers}$：Transformer层数
- $n_{heads}$：注意力头数
- $d_{head}$：每个头的维度
- $L$：序列长度
- $b_{bytes}$：每个数值的字节数（FP16为2字节）
- 系数2：Key和Value各一份

以LLaMA-7B为例（32层、32头、128维、FP16），序列长度4096时KV Cache约需3.5GB，超过模型权重本身。长序列推理的内存瓶颈往往是KV Cache而非模型权重。

## KV Cache优化的目标

1. **降低内存占用**：量化、压缩、淘汰不重要的KV
2. **高效内存管理**：分页管理、前缀复用
3. **限制缓存增长**：滑动窗口、注意力淘汰

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
from collections import OrderedDict

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")

---
## 2.1.1 KV Cache 量化

### 什么是KV Cache量化？

将缓存的Key/Value张量从FP16量化为INT8或INT4，直接将KV Cache内存占用减半或降至1/4。由于Key和Value的分布不同，通常对Key使用逐通道量化，对Value使用逐张量量化。

### 为什么KV Cache量化有效？

1. **KV Cache是内存瓶颈**：长序列推理时KV Cache占用的内存远超模型权重，量化KV Cache可以显著降低内存占用和带宽需求。
2. **KV值对量化不敏感**：研究表明，注意力机制对KV的精度有一定容忍度，INT8量化几乎无损，INT4量化损失也很小。
3. **带宽节省**：推理的瓶颈往往是内存带宽，量化后数据搬运量减半/降至1/4，直接提升推理速度。

### KV Cache量化的数学公式

**Key量化（逐通道）**：
$$s_{k,c} = \frac{\max(|K_{:,c}|)}{q_{max}}, \quad K_{q,:,c} = \text{round}(K_{:,c} / s_{k,c})$$

**Value量化（逐张量）**：
$$s_v = \frac{\max(|V|)}{q_{max}}, \quad V_q = \text{round}(V / s_v)$$

其中：
- $K_{:,c}$：Key张量第$c$个通道的所有元素
- $s_{k,c}$：Key第$c$个通道的缩放因子
- $s_v$：Value的全局缩放因子
- $q_{max}$：量化最大值（INT8为127，INT4为7）
- Key用逐通道量化是因为不同通道的数值范围差异大
- Value用逐张量量化是因为其分布更均匀

In [ ]:
class KVCacheQuantizer:
    """KV Cache量化器"""
    def __init__(self, n_bits=8):
        self.n_bits = n_bits
        self.q_max = 2 ** (n_bits - 1) - 1
        self.q_min = -2 ** (n_bits - 1)

    def quantize_kv(self, key: torch.Tensor, value: torch.Tensor) -> tuple:
        """量化KV Cache
        Key: 逐通道量化（每个注意力头每个通道独立scale）
        Value: 逐张量量化（整个张量共享scale）
        """
        k_scale = key.abs().amax(dim=(0, 1), keepdim=True) / self.q_max
        k_scale = torch.clamp(k_scale, min=1e-8)
        k_q = torch.clamp(torch.round(key / k_scale), self.q_min, self.q_max).to(torch.int8)

        v_scale = value.abs().max() / self.q_max
        v_scale = torch.clamp(v_scale, min=1e-8)
        v_q = torch.clamp(torch.round(value / v_scale), self.q_min, self.q_max).to(torch.int8)

        return k_q, k_scale, v_q, v_scale

    def dequantize_kv(self, k_q, k_scale, v_q, v_scale):
        """反量化KV Cache"""
        key_deq = k_q.float() * k_scale
        value_deq = v_q.float() * v_scale
        return key_deq, value_deq

batch_size, n_heads, seq_len, head_dim = 1, 8, 512, 64
key = torch.randn(batch_size, n_heads, seq_len, head_dim)
value = torch.randn(batch_size, n_heads, seq_len, head_dim)

kv_fp16_bytes = key.numel() * 2 + value.numel() * 2

quantizer_8bit = KVCacheQuantizer(n_bits=8)
k_q8, k_s8, v_q8, v_s8 = quantizer_8bit.quantize_kv(key, value)
k_deq8, v_deq8 = quantizer_8bit.dequantize_kv(k_q8, k_s8, v_q8, v_s8)
kv_int8_bytes = k_q8.numel() * 1 + v_q8.numel() * 1 + k_s8.numel() * 2 + v_s8.numel() * 2

quantizer_4bit = KVCacheQuantizer(n_bits=4)
k_q4, k_s4, v_q4, v_s4 = quantizer_4bit.quantize_kv(key, value)
k_deq4, v_deq4 = quantizer_4bit.dequantize_kv(k_q4, k_s4, v_q4, v_s4)
kv_int4_bytes = k_q4.numel() * 0.5 + v_q4.numel() * 0.5 + k_s4.numel() * 2 + v_s4.numel() * 2

k_err8 = (key - k_deq8).norm() / key.norm() * 100
v_err8 = (value - v_deq8).norm() / value.norm() * 100
k_err4 = (key - k_deq4).norm() / key.norm() * 100
v_err4 = (value - v_deq4).norm() / value.norm() * 100

print(f"=== KV Cache量化效果 ===")
print(f"序列长度={seq_len}, 头数={n_heads}, 头维度={head_dim}")
print(f"\n{'格式':<15} {'内存(KB)':<15} {'Key误差%':<12} {'Value误差%':<12}")
print("-" * 54)
print(f"{'FP16':<15} {kv_fp16_bytes/1024:<15.2f} {'0.00':<12} {'0.00':<12}")
print(f"{'INT8':<15} {kv_int8_bytes/1024:<15.2f} {k_err8:<12.4f} {v_err8:<12.4f}")
print(f"{'INT4':<15} {kv_int4_bytes/1024:<15.2f} {k_err4:<12.4f} {v_err4:<12.4f}")
print(f"\n压缩比: FP16->INT8={kv_fp16_bytes/kv_int8_bytes:.2f}x, FP16->INT4={kv_fp16_bytes/kv_int4_bytes:.2f}x")

for sl in [256, 512, 1024, 2048, 4096]:
    fp16_mem = batch_size * n_heads * sl * head_dim * 2 * 2 / 1024
    int8_mem = batch_size * n_heads * sl * head_dim * 1 * 2 / 1024
    int4_mem = batch_size * n_heads * sl * head_dim * 0.5 * 2 / 1024
    print(f"seq_len={sl:<5} FP16={fp16_mem:.1f}KB INT8={int8_mem:.1f}KB INT4={int4_mem:.1f}KB")

---
## 2.1.2 KV Cache 内存管理

### PagedAttention（vLLM）

#### 什么是PagedAttention？

PagedAttention借鉴操作系统虚拟内存的分页管理思想，将KV Cache划分为固定大小的block，按需分配，非连续存储。vLLM推理框架的核心技术。

#### 为什么PagedAttention有效？

1. **消除内存碎片**：传统KV Cache需要为每个序列预分配最大长度的连续内存，造成大量浪费。PagedAttention按block分配，无需连续内存。
2. **支持更大batch**：内存利用率从传统方式的60%-80%提升到接近100%，可服务更多并发请求。
3. **动态内存管理**：序列完成即可释放block，新序列可复用，类似OS的内存分页机制。

#### PagedAttention的数学原理

传统方式为每个序列预分配最大长度$L_{max}$的连续内存：
$$M_{traditional} = B \times L_{max} \times d_{kv} \times b_{bytes}$$

PagedAttention按block分配，实际使用量等于真实序列长度：
$$M_{paged} = \sum_{i=1}^{B} \lceil L_i / B_s \rceil \times B_s \times d_{kv} \times b_{bytes}$$

其中：
- $B$：batch大小（并发请求数）
- $L_{max}$：最大序列长度
- $L_i$：第$i$个请求的实际序列长度
- $B_s$：block大小（通常16个token）
- $d_{kv}$：KV向量的维度
- 浪费率仅取决于block内的碎片（最多$B_s - 1$个token的浪费）

In [ ]:
class PagedKVCache:
    """PagedAttention简化实现"""
    def __init__(self, n_heads, head_dim, block_size=16, n_blocks=64):
        self.n_heads = n_heads
        self.head_dim = head_dim
        self.block_size = block_size
        self.n_blocks = n_blocks
        self.block_table = {}
        self.free_blocks = list(range(n_blocks))
        self.kv_block_pool = torch.zeros(n_blocks, 2, n_heads, block_size, head_dim)
        self.seq_block_mapping = {}

    def allocate_block(self):
        if not self.free_blocks:
            return None
        block_id = self.free_blocks.pop(0)
        return block_id

    def free_block(self, block_id):
        self.free_blocks.append(block_id)
        self.kv_block_pool[block_id].zero_()

    def append_kv(self, seq_id, key, value):
        """向指定序列追加KV对"""
        if seq_id not in self.seq_block_mapping:
            self.seq_block_mapping[seq_id] = []

        blocks = self.seq_block_mapping[seq_id]
        if not blocks or self._get_block_usage(seq_id) >= self.block_size:
            new_block = self.allocate_block()
            if new_block is None:
                raise RuntimeError("KV Cache OOM: no free blocks")
            blocks.append(new_block)

        current_block = blocks[-1]
        usage = self._get_block_usage(seq_id)
        self.kv_block_pool[current_block, 0, :, usage, :] = key
        self.kv_block_pool[current_block, 1, :, usage, :] = value

    def _get_block_usage(self, seq_id):
        total_tokens = sum(1 for _ in range(self._count_tokens(seq_id)))
        if self.seq_block_mapping[seq_id]:
            return total_tokens % self.block_size if total_tokens % self.block_size != 0 else 0
        return 0

    def _count_tokens(self, seq_id):
        if seq_id not in self.seq_block_mapping or not self.seq_block_mapping[seq_id]:
            return range(0)
        n_blocks = len(self.seq_block_mapping[seq_id])
        return range(n_blocks * self.block_size)

    def get_kv(self, seq_id):
        """获取指定序列的所有KV"""
        if seq_id not in self.seq_block_mapping:
            return None, None
        blocks = self.seq_block_mapping[seq_id]
        keys = [self.kv_block_pool[b, 0] for b in blocks]
        values = [self.kv_block_pool[b, 1] for b in blocks]
        return torch.cat(keys, dim=1), torch.cat(values, dim=1)

    def memory_usage(self):
        used = self.n_blocks - len(self.free_blocks)
        return used / self.n_blocks * 100

paged_cache = PagedKVCache(n_heads=8, head_dim=64, block_size=16, n_blocks=64)

for step in range(32):
    k = torch.randn(1, 8, 1, 64)
    v = torch.randn(1, 8, 1, 64)
    paged_cache.append_kv(seq_id=0, key=k.squeeze(2), value=v.squeeze(2))

for step in range(16):
    k = torch.randn(1, 8, 1, 64)
    v = torch.randn(1, 8, 1, 64)
    paged_cache.append_kv(seq_id=1, key=k.squeeze(2), value=v.squeeze(2))

keys_0, values_0 = paged_cache.get_kv(seq_id=0)
keys_1, values_1 = paged_cache.get_kv(seq_id=1)

print(f"=== PagedAttention ===")
print(f"Block大小: 16 tokens")
print(f"总Block数: 64")
print(f"序列0: {keys_0.shape[1]} tokens, 使用{len(paged_cache.seq_block_mapping[0])}个block")
print(f"序列1: {keys_1.shape[1]} tokens, 使用{len(paged_cache.seq_block_mapping[1])}个block")
print(f"内存使用率: {paged_cache.memory_usage():.1f}%")
print(f"\nPagedAttention核心优势:")
print(f"1. 非连续内存分配，消除碎片")
print(f"2. 按需分配，不同序列可共享block池")
print(f"3. 支持动态batch，序列完成即可释放block")

### Prefix Caching

#### 什么是Prefix Caching？

对共享前缀（如system prompt）的KV Cache进行缓存复用，避免重复计算。多个请求共享相同前缀时，只需计算一次前缀的KV，后续请求直接复用。

#### 为什么Prefix Caching有效？

1. **前缀复用率高**：在对话场景中，system prompt通常占数百个token，且所有请求共享，缓存后可节省大量重复计算。
2. **零精度损失**：完全复用已计算的精确KV，不引入任何近似误差。
3. **延迟降低**：新请求跳过前缀的prefill阶段，首token延迟显著降低。

#### Prefix Caching的数学分析

设前缀长度为$L_p$，用户输入平均长度为$L_u$，则：
- 无缓存：每个请求需计算$(L_p + L_u)$个token的KV
- 有缓存：每个请求只需计算$L_u$个token的KV
- 计算节省比：$\frac{L_p}{L_p + L_u}$

当$L_p = 512, L_u = 64$时，节省比约89%。

In [ ]:
class PrefixCache:
    """前缀缓存：复用共享前缀的KV Cache"""
    def __init__(self, max_cache_size=100):
        self.cache = OrderedDict()
        self.max_cache_size = max_cache_size

    def get_prefix_kv(self, prefix_hash):
        """获取缓存的前缀KV"""
        if prefix_hash in self.cache:
            self.cache.move_to_end(prefix_hash)
            return self.cache[prefix_hash]
        return None

    def put_prefix_kv(self, prefix_hash, kv_data):
        """缓存前缀KV"""
        if prefix_hash in self.cache:
            self.cache.move_to_end(prefix_hash)
        self.cache[prefix_hash] = kv_data
        if len(self.cache) > self.max_cache_size:
            self.cache.popitem(last=False)

    def compute_prefix_hash(self, token_ids):
        """计算前缀的哈希值"""
        return hash(tuple(token_ids.tolist()))

prefix_cache = PrefixCache(max_cache_size=50)

system_prompt_tokens = torch.randint(0, 1000, (128,))
prefix_hash = prefix_cache.compute_prefix_hash(system_prompt_tokens)

system_k = torch.randn(1, 8, 128, 64)
system_v = torch.randn(1, 8, 128, 64)
prefix_cache.put_prefix_kv(prefix_hash, (system_k, system_v))

n_requests = 5
cache_hits = 0
total_compute_tokens = 0
saved_tokens = 0

for i in range(n_requests):
    user_tokens = torch.randint(0, 1000, (64 + i * 10,))
    full_tokens = torch.cat([system_prompt_tokens, user_tokens])
    total_compute_tokens += len(full_tokens)

    cached = prefix_cache.get_prefix_kv(prefix_hash)
    if cached is not None:
        cache_hits += 1
        saved_tokens += len(system_prompt_tokens)

print(f"=== Prefix Caching效果 ===")
print(f"请求数: {n_requests}")
print(f"缓存命中: {cache_hits}/{n_requests}")
print(f"总需计算tokens: {total_compute_tokens}")
print(f"缓存节省tokens: {saved_tokens}")
print(f"计算量节省: {saved_tokens/total_compute_tokens*100:.1f}%")
print(f"\nPrefix Caching特别适合: 多用户共享system prompt的对话场景")

---
## 2.1.3 KV Cache 压缩与淘汰

### 滑动窗口注意力（Sliding Window Attention）

#### 什么是滑动窗口注意力？

每个token只关注最近$W$个token的KV，超出窗口的KV被丢弃。KV Cache大小固定为$O(W)$，不随序列增长。Mistral、Gemini等模型采用此策略。

#### 为什么滑动窗口有效？

1. **内存可控**：KV Cache大小固定为$W$，不随序列长度增长，适合端侧设备的有限内存。
2. **局部性原理**：自然语言中，相邻token的依赖关系最强，远距离依赖相对较少，窗口注意力在大多数任务上表现良好。
3. **信息传递**：虽然每个token只看$W$个邻居，但通过多层堆叠，信息可以逐层传递到更远的token（感受野为$W \times L$）。

#### 滑动窗口的数学公式

注意力计算仅对窗口内的KV进行：
$$\text{Attn}(q_i, K, V) = \text{softmax}\left(\frac{q_i K_{[i-W:i]}^T}{\sqrt{d_k}}\right) V_{[i-W:i]}$$

其中：
- $q_i$：第$i$个token的查询向量
- $K_{[i-W:i]}$：第$i-W$到第$i$个token的Key矩阵
- $V_{[i-W:i]}$：对应的Value矩阵
- $W$：窗口大小
- $d_k$：Key的维度
- 多层后的有效感受野：$W \times n_{layers}$

In [ ]:
class SlidingWindowKVCache:
    """滑动窗口KV Cache"""
    def __init__(self, n_heads, head_dim, window_size=256):
        self.n_heads = n_heads
        self.head_dim = head_dim
        self.window_size = window_size
        self.key_cache = torch.zeros(1, n_heads, window_size, head_dim)
        self.value_cache = torch.zeros(1, n_heads, window_size, head_dim)
        self.current_pos = 0

    def update(self, new_key, new_value):
        """更新KV Cache，超出窗口自动淘汰最旧的KV"""
        seq_len = new_key.shape[2]
        for i in range(seq_len):
            pos = self.current_pos % self.window_size
            self.key_cache[:, :, pos, :] = new_key[:, :, i, :]
            self.value_cache[:, :, pos, :] = new_value[:, :, i, :]
            self.current_pos += 1

    def get_kv(self):
        """获取当前窗口内的KV（保证最新token在末尾）"""
        if self.current_pos <= self.window_size:
            return (self.key_cache[:, :, :self.current_pos, :],
                    self.value_cache[:, :, :self.current_pos, :])
        start = self.current_pos % self.window_size
        key_ordered = torch.cat([self.key_cache[:, :, start:, :], self.key_cache[:, :, :start, :]], dim=2)
        value_ordered = torch.cat([self.value_cache[:, :, start:, :], self.value_cache[:, :, :start, :]], dim=2)
        return key_ordered, value_ordered

    def memory_mb(self):
        return self.key_cache.numel() * 2 / 1024 / 1024 + self.value_cache.numel() * 2 / 1024 / 1024

n_heads, head_dim = 8, 64
window_sizes = [128, 256, 512, 1024]

print(f"=== 滑动窗口KV Cache ===")
print(f"\n内存占用对比 (固定，不随序列增长):")
for ws in window_sizes:
    sw_cache = SlidingWindowKVCache(n_heads, head_dim, window_size=ws)
    print(f"  窗口={ws:<6} 内存={sw_cache.memory_mb():.3f} MB")

sw = SlidingWindowKVCache(n_heads, head_dim, window_size=256)
for step in range(500):
    k = torch.randn(1, n_heads, 1, head_dim)
    v = torch.randn(1, n_heads, 1, head_dim)
    sw.update(k, v)

k_out, v_out = sw.get_kv()
print(f"\n写入500个token后:")
print(f"  实际缓存KV长度: {k_out.shape[2]}")
print(f"  内存占用: {sw.memory_mb():.3f} MB (恒定)")
print(f"  丢弃的旧KV: {500 - 256} tokens")

print(f"\n对比无限制KV Cache:")
for seq_len in [256, 512, 1024, 2048, 4096]:
    unlimited_mem = 1 * n_heads * seq_len * head_dim * 2 * 2 / 1024 / 1024
    sw_mem = 1 * n_heads * 256 * head_dim * 2 * 2 / 1024 / 1024
    print(f"  seq={seq_len:<5} 无限制={unlimited_mem:.3f}MB 滑动窗口(256)={sw_mem:.3f}MB 节省{(1-sw_mem/unlimited_mem)*100:.0f}%")

### H2O - 基于注意力分数的KV淘汰

#### 什么是H2O？

H2O（Heavy-Hitter Oracle）当KV Cache超出预算时，根据注意力分数选择性淘汰不重要的KV。保留注意力分数最高的KV（Heavy Hitter）和最近的KV。

#### 为什么H2O有效？

1. **注意力分布不均**：LLM的注意力分数呈长尾分布，少数token获得大部分注意力（Heavy Hitter），这些token对输出影响最大。
2. **保留重要信息**：淘汰注意力分数低的KV，保留对输出影响最大的KV，精度损失最小。
3. **最近token保护**：最近的token通常对下一个token的预测最重要，必须保留。

#### H2O的数学原理

累积注意力分数作为重要性度量：
$$S_i = \sum_{t=i+1}^{L} A_{t,i}$$

淘汰策略：在预算$B$内，保留：
- 最近$W$个token的KV（recent window）
- 历史KV中累积注意力分数最高的$B - W$个token

其中：
- $S_i$：第$i$个token的累积注意力分数
- $A_{t,i}$：第$t$个token对第$i$个token的注意力权重
- $B$：KV Cache预算（保留的最大token数）
- $W$：最近窗口大小
- $L$：当前序列长度

In [ ]:
class H2OKVCache:
    """H2O: 基于注意力分数的KV淘汰策略"""
    def __init__(self, n_heads, head_dim, budget=128, recent_window=32):
        self.n_heads = n_heads
        self.head_dim = head_dim
        self.budget = budget
        self.recent_window = recent_window
        self.keys = []
        self.values = []
        self.attention_scores = []

    def update(self, new_key, new_value, attn_weights=None):
        """更新KV Cache"""
        self.keys.append(new_key)
        self.values.append(new_value)
        if attn_weights is not None:
            self.attention_scores.append(attn_weights.sum(dim=(0, 1)))
        else:
            self.attention_scores.append(torch.tensor(1.0))

        if len(self.keys) > self.budget:
            self._evict()

    def _evict(self):
        """淘汰策略：保留最近窗口内的 + 注意力分数最高的"""
        n = len(self.keys)
        recent_start = max(0, n - self.recent_window)

        older_indices = list(range(recent_start))
        if older_indices:
            older_scores = torch.tensor([self.attention_scores[i] for i in older_indices])
            n_to_keep = self.budget - self.recent_window
            n_to_keep = min(n_to_keep, len(older_indices))
            _, top_indices = older_scores.topk(n_to_keep)
            keep_set = set(older_indices[i] for i in top_indices.tolist())
            keep_set.update(range(recent_start, n))

            self.keys = [self.keys[i] for i in sorted(keep_set)]
            self.values = [self.values[i] for i in sorted(keep_set)]
            self.attention_scores = [self.attention_scores[i] for i in sorted(keep_set)]

    def get_kv(self):
        if not self.keys:
            return None, None
        return torch.cat(self.keys, dim=2), torch.cat(self.values, dim=2)

    def __len__(self):
        return len(self.keys)

h2o = H2OKVCache(n_heads=8, head_dim=64, budget=128, recent_window=32)
unlimited_keys = []

for step in range(256):
    k = torch.randn(1, 8, 1, 64)
    v = torch.randn(1, 8, 1, 64)
    attn_score = torch.rand(1) * 10
    h2o.update(k, v, attn_score)
    unlimited_keys.append(k)

k_out, v_out = h2o.get_kv()
unlimited_mem = 1 * 8 * 256 * 64 * 2 * 2 / 1024 / 1024
h2o_mem = 1 * 8 * len(h2o) * 64 * 2 * 2 / 1024 / 1024

print(f"=== H2O KV淘汰策略 ===")
print(f"总生成tokens: 256")
print(f"KV Cache预算: 128")
print(f"实际缓存tokens: {len(h2o)}")
print(f"无限制内存: {unlimited_mem:.3f} MB")
print(f"H2O内存: {h2o_mem:.3f} MB")
print(f"节省: {(1-h2o_mem/unlimited_mem)*100:.0f}%")
print(f"\nH2O策略: 保留最近32个tokens + 注意力分数最高的96个tokens")

### 跨层KV共享（Cross-Layer KV Sharing）

#### 什么是跨层KV共享？

相邻层的KV表示高度相似，可共享同一份KV Cache。如CLA（Cross-Layer Attention）、YOCO等架构，将KV Cache占用减半。

#### 为什么跨层KV共享有效？

1. **层间相似性**：研究表明Transformer相邻层的Key和Value向量余弦相似度高达0.9以上，存在大量冗余。
2. **内存直接减半**：每2层共享1组KV投影，KV Cache减少50%，且无需额外压缩操作。
3. **计算量也减少**：KV投影层减少一半，前向计算量也相应降低。

#### 跨层KV共享的数学原理

标准Transformer每层独立计算KV：
$$K_l = X_l W_k^l, \quad V_l = X_l W_v^l, \quad l = 1, 2, ..., L$$

跨层共享策略（每$s$层共享1组KV投影）：
$$K_l = X_l W_k^{\lfloor l/s \rfloor}, \quad V_l = X_l W_v^{\lfloor l/s \rfloor}$$

其中：
- $X_l$：第$l$层的输入
- $W_k^l, W_v^l$：第$l$层的Key和Value投影矩阵
- $s$：共享步长（stride），通常为2
- $\lfloor l/s \rfloor$：整除，表示第$l$层使用的KV投影组索引
- KV投影参数从$L$组减少到$L/s$组

In [ ]:
class CrossLayerKVSharedTransformer(nn.Module):
    """跨层KV共享的Transformer"""
    def __init__(self, n_layers=6, dim=64, n_heads=4, share_stride=2):
        super().__init__()
        self.n_layers = n_layers
        self.share_stride = share_stride
        self.q_projs = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_layers)])
        self.kv_projs = nn.ModuleList([nn.Linear(dim, 2 * dim) for _ in range(n_layers // share_stride)])
        self.out_projs = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_layers)])
        self.ffns = nn.ModuleList([nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim)) for _ in range(n_layers)])
        self.lns = nn.ModuleList([nn.LayerNorm(dim) for _ in range(n_layers * 2)])

    def _get_kv_layer_idx(self, layer_idx):
        return layer_idx // self.share_stride

    def forward(self, x):
        B, N, C = x.shape
        head_dim = C // 4
        for i in range(self.n_layers):
            h = self.lns[2*i](x)
            q = self.q_projs[i](h).view(B, N, 4, head_dim).transpose(1, 2)
            kv_idx = self._get_kv_layer_idx(i)
            kv = self.kv_projs[kv_idx](h)
            k, v = kv.chunk(2, dim=-1)
            k = k.view(B, N, 4, head_dim).transpose(1, 2)
            v = v.view(B, N, 4, head_dim).transpose(1, 2)
            attn = (q @ k.transpose(-2, -1)) * (head_dim ** -0.5)
            attn = attn.softmax(dim=-1)
            out = (attn @ v).transpose(1, 2).reshape(B, N, C)
            out = self.out_projs[i](out)
            x = x + out
            x = x + self.ffns[i](self.lns[2*i+1](x))
        return x

n_layers = 6
model_shared = CrossLayerKVSharedTransformer(n_layers=n_layers, dim=64, n_heads=4, share_stride=2)

standard_kv_layers = n_layers
shared_kv_layers = n_layers // 2

seq_len = 512
kv_per_layer = 1 * 4 * seq_len * 64 * 2 * 2

standard_kv_mem = standard_kv_layers * kv_per_layer / 1024 / 1024
shared_kv_mem = shared_kv_layers * kv_per_layer / 1024 / 1024

x = torch.randn(2, 16, 64)
out = model_shared(x)

print(f"=== 跨层KV共享 ===")
print(f"层数: {n_layers}")
print(f"共享策略: 每2层共享1组KV投影")
print(f"标准Transformer KV层数: {standard_kv_layers}")
print(f"共享后KV层数: {shared_kv_layers}")
print(f"\nKV Cache内存:")
print(f"  标准Transformer: {standard_kv_mem:.3f} MB")
print(f"  跨层共享: {shared_kv_mem:.3f} MB")
print(f"  节省: {(1-shared_kv_mem/standard_kv_mem)*100:.0f}%")
print(f"\n模型输出形状: {out.shape} (正常工作)")
print(f"\n跨层KV共享前提: 相邻层的Key/Value表示高度相似")

### KV Cache优化方法综合对比

不同KV Cache优化方法在内存节省、精度影响和适用场景方面的对比。实际部署中通常组合使用多种方法。

In [ ]:
print(f"{'方法':<25} {'内存节省':<15} {'精度影响':<15} {'适用场景':<25}")
print("-" * 80)
methods = [
    ("KV INT8量化", "50%", "极小", "通用"),
    ("KV INT4量化", "75%", "较小", "长序列/内存紧张"),
    ("PagedAttention", "消除碎片", "无", "多请求并发"),
    ("Prefix Caching", "共享前缀100%", "无", "多用户共享prompt"),
    ("滑动窗口", "固定O(W)", "丢失远距依赖", "短上下文/流式"),
    ("H2O淘汰", "可控", "较小", "长序列/预算受限"),
    ("跨层KV共享", "50%", "需验证", "层间相似度高"),
]
for name, saving, impact, scene in methods:
    print(f"{name:<25} {saving:<15} {impact:<15} {scene:<25}")

print(f"\n=== 产业级组合建议 ===")
print(f"1. 量化 + PagedAttention: 最通用的组合，vLLM默认方案")
print(f"2. 量化 + 滑动窗口: 端侧设备首选，内存可控")
print(f"3. Prefix Caching + 量化: 对话服务场景，复用system prompt")
print(f"4. H2O + 量化: 长文本生成，在有限预算内保留最重要KV")